In [ ]:
# ============================================================
# 環境設定（Colab/ローカル共通）
# ============================================================
import os

# Google Earth Engineプロジェクト ID（ご自身のGEEプロジェクトIDに変更）
GEE_PROJECT = os.environ.get('GEE_PROJECT', 'your-ee-project-id')

# 出力ディレクトリ（Colabの場合は/content/drive/MyDrive/...、ローカルの場合は任意のパス）
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', '/content/drive/MyDrive/Downscaling')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'GEE_PROJECT: {GEE_PROJECT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


In [2]:
# STEP 1: Import and initialize
import ee
import numpy as np
import pandas as pd
from google.colab import files

ee.Authenticate()
import os
GEE_PROJECT = os.environ.get('GEE_PROJECT', 'your-ee-project')
ee.Initialize(project=GEE_PROJECT)


In [3]:
# STEP 2: Parameters
region = ee.Geometry.Rectangle([71, 17, 75, 20])
start_date = '1986-01-01'
end_date = '2006-01-01'
seconds = 24 * 3600
variables = ['pr', 'tas', 'rlds', 'rsds', 'sfcWind']


In [4]:
# STEP 3: Load observation data (GSMaP + ERA5)
#gsmap = ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational") \
    #.filterDate(start_date, end_date).filterBounds(region).select('hourlyPrecipRateGC')
#OBS_pr = gsmap.mean().multiply(24).clip(region).rename('pr')

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
    .filterDate(start_date, end_date).filterBounds(region)
OBS_pr = era5.select('total_precipitation_sum').mean().multiply(1000).clip(region).rename('pr')  # mm/da
OBS_tas = era5.select('temperature_2m').mean().clip(region).rename('tas')
OBS_rsds = era5.select('surface_net_solar_radiation_sum').mean().divide(seconds).clip(region).rename('rsds')
OBS_rlds = era5.select('surface_thermal_radiation_downwards_sum').mean().divide(seconds).clip(region).rename('rlds')
era5_u = era5.select('u_component_of_wind_10m').mean()
era5_v = era5.select('v_component_of_wind_10m').mean()
OBS_sfcWind = era5_u.pow(2).add(era5_v.pow(2)).sqrt().clip(region).rename('sfcWind')

obs_dict = {
    'pr': OBS_pr,
    'tas': OBS_tas,
    'rlds': OBS_rlds,
    'rsds': OBS_rsds,
    'sfcWind': OBS_sfcWind
}

In [5]:
# STEP 4: Get GCM model list
gcm_base = ee.ImageCollection("NASA/GDDP-CMIP6") \
    .filter(ee.Filter.eq('scenario', 'historical')).filterDate(start_date, end_date)
models = gcm_base.aggregate_histogram('model').keys().getInfo()

In [ ]:
# STEP 6: Analysis
results = []

for model in models:
    print(f"Processing model: {model}")
    model_result = {'model': model}

    try:
        gcm = gcm_base.filter(ee.Filter.eq('model', model))
        gcm_proj = gcm.first().select('pr').projection()
        scale = gcm_proj.nominalScale()

        for var in variables:
            obs_img = obs_dict[var].reproject(crs=gcm_proj.crs(), scale=scale).clip(region)
            gcm_img = gcm.select(var).mean()
            if var == 'pr':
                gcm_img = gcm_img.multiply(86400)
            gcm_img = gcm_img.reproject(crs=gcm_proj.crs(), scale=scale).clip(region)

            # Sampling with IDs
            def add_id(feature):
                coords = feature.geometry().coordinates()
                lon = ee.Number(coords.get(0)).format('%.4f')
                lat = ee.Number(coords.get(1)).format('%.4f')
                return feature.set('id', lon.cat('_').cat(lat))

            obs_pts = obs_img.sample(region=region, scale=scale, geometries=True).map(add_id)
            gcm_pts = gcm_img.sample(region=region, scale=scale, geometries=True).map(add_id)

            joined = ee.Join.inner().apply(
                obs_pts, gcm_pts,
                ee.Filter.equals(leftField='id', rightField='id')
            )

            formatted = joined.map(lambda f: ee.Feature(None, {
                'id': ee.Feature(f.get('primary')).get('id'),
                'obs': ee.Feature(f.get('primary')).get(var),
                'gcm': ee.Feature(f.get('secondary')).get(var)
            }))

            # Extract and analyze
            ids = formatted.aggregate_array('id').getInfo()
            obs_vals = formatted.aggregate_array('obs').getInfo()
            gcm_vals = formatted.aggregate_array('gcm').getInfo()

            if len(ids) == len(obs_vals) == len(gcm_vals) and len(ids) > 0:
                df = pd.DataFrame({'id': ids, 'obs': obs_vals, 'gcm': gcm_vals}).dropna()
                corr = np.corrcoef(df['obs'], df['gcm'])[0, 1]
                rmse = np.sqrt(np.mean((df['obs'] - df['gcm']) ** 2))
            else:
                corr = np.nan
                rmse = np.nan

            model_result[f"{var}_corr"] = corr
            model_result[f"{var}_rmse"] = rmse

    except Exception as e:
        print(f"Error in {model}: {e}")
        for var in variables:
            model_result[f"{var}_corr"] = np.nan
            model_result[f"{var}_rmse"] = np.nan

    results.append(model_result)


Processing model: ACCESS-CM2
Processing model: ACCESS-ESM1-5
Processing model: BCC-CSM2-MR
Processing model: CESM2
Processing model: CESM2-WACCM
Processing model: CMCC-CM2-SR5
Processing model: CMCC-ESM2
Processing model: CNRM-CM6-1
Processing model: CNRM-ESM2-1
Processing model: CanESM5
Processing model: EC-Earth3
Processing model: EC-Earth3-Veg-LR
Processing model: FGOALS-g3
Processing model: GFDL-CM4
Processing model: GFDL-ESM4
Processing model: GISS-E2-1-G
Processing model: HadGEM3-GC31-LL
Processing model: HadGEM3-GC31-MM
Processing model: IITM-ESM
Processing model: INM-CM4-8
Processing model: INM-CM5-0


Processing model: IPSL-CM6A-LR
Processing model: KACE-1-0-G


In [ ]:
# STEP 7: Export
df = pd.DataFrame(results)
df.to_csv("GCMs_Evaluation.csv", index=False)
df
files.download("GCMs_Evaluation.csv")
